# SI4006 · Entrega M2 — Harness de evaluación y scorecard del baseline
### Equipo Offside — Jean Carlo Londoño Ocampo · Alejandro Garcés Ramírez · Nicolás Ospina Torres

**Qué evalúa este harness:** si la señal que Offside le entrega a un apostador —qué pasó, a quién le
afecta y cuánto— le sirve para decidir mejor que si hubiera leído el titular por su cuenta.

**Qué es una buena respuesta en nuestro dominio:** una que **acierta el tipo de hecho, no se equivoca
en la dirección del impacto, no confunde un rumor con un hecho, y ante la duda calla en vez de
inventar**.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JeanCarloLond/offside-londono-garces-ospina/blob/main/proyecto1/notebooks/M2_harness_evaluacion.ipynb)

Al abrirlo: Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU**, y luego Ejecutar todas.

> **Procedencia de los outputs.** Este notebook se corrió de principio a fin en CPU el 2026-08-27
> para obtener números reales; los outputs guardados son los de esa corrida. En Colab con T4 el juez
> tarda segundos en vez de ~17 s por ejemplo. El juez es determinista (se leen logits, no se genera
> texto), así que los números no cambian por hardware.

## 0 · Setup

Semilla fija (`SEED = 42`). Instalamos solo lo que Colab no trae.

In [ ]:
%pip install -q peft datasets accelerate pyyaml scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!git clone --depth 1 https://github.com/JeanCarloLond/offside-londono-garces-ospina.git offside

Cloning into 'offside'...


In [ ]:
import json
import os
import random
import sys

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.chdir("offside/proyecto1/eval")
sys.path.insert(0, os.getcwd())

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("cwd:", os.getcwd())

device: cpu
cwd: /content/offside/proyecto1/eval


## 1 · El eval set de dominio

21 ejemplos curados a mano, cada uno con **`input`**, **`esperado`** y **`criterio`**
(qué haría buena a la respuesta en ese caso concreto).

**13 de 21 (61 %) son adversariales o de borde** — el
mínimo que pide el módulo es 20 %. Son trampas, casos ambiguos, uno fuera de dominio (tenis) y un
rumor. Hay además 3 controles fáciles a propósito: sin ellos no se puede distinguir «el sistema es
malo» de «los casos son imposibles».

No está contaminado: son noticias de agosto de 2026 etiquetadas por nosotros, no salen de ningún
benchmark público.

In [ ]:
eval_set = json.load(open("eval_set.json", encoding="utf-8"))
adv = [e for e in eval_set if e["adversarial"]]

print(
    f"ejemplos: {len(eval_set)} | adversariales: {len(adv)} ({100 * len(adv) / len(eval_set):.0f}%)"
)
print(f"categorias cubiertas: {len({e['esperado']['category'] for e in eval_set})}/8")
print()
ej = eval_set[10]
print("EJEMPLO —", ej["eval_id"], "|", ej["tipo_dificultad"])
print("  input   :", ej["input"]["text"][:100], "...")
print("  esperado:", ej["esperado"])
print("  criterio:", ej["criterio"][:150], "...")

ejemplos: 21 | adversariales: 13 (62%)
categorias cubiertas: 8/8

EJEMPLO — OFF-11 | trampa: par mínimo con OFF-03
  input   : El VAR castiga al Celta y Osasuna lo remata. Una rigurosa roja directa a Marcos Alonso vista desde e ...
  esperado: {'category': 'sancion_suspension', 'impact': 'negativo_alto', 'team': 'Celta de Vigo'}
  criterio: Aquí la roja SÍ cuenta: es un partido de liga, así que arrastra sanción para la jornada siguiente y el Celta pierde a ese jugador. La respuesta buena  ...


### El par mínimo que más nos gusta

`OFF-03` y `OFF-11` son casi el mismo texto —una roja directa— y tienen la respuesta **opuesta**:
la primera fue en un amistoso de pretemporada (no arrastra sanción, así que no hay señal que dar) y
la segunda en partido de liga (sí la arrastra). Lo único que las separa es el contexto de
competición, y no está dicho de forma explícita.

## 2 · La rúbrica del juez (versionada)

Está en `judge_rubric.yaml`. Cambiar un ancla cambia el veredicto, así que el scorecard solo es
comparable entre corridas con la misma versión — el harness la registra.

El orden de la escala no es «cuánto acierta» sino **cuánto daño hace al decidir**: por eso *no emitir
señal* (3) puntúa por encima de *emitir una falsa* (2).

In [ ]:
import yaml

rub = yaml.safe_load(open("judge_rubric.yaml", encoding="utf-8"))
print(f"rúbrica versión {rub['version']}\n")
for a in sorted(rub["anclas"], key=lambda x: -x["nivel"]):
    print(f"{a['nivel']} = {a['nombre']}")
    print(f"    {' '.join(a['descripcion'].split())[:160]}...")

rúbrica versión 1

5 = Señal correcta y accionable
    Acierta el tipo de hecho, la dirección del impacto (negativo / neutro / positivo) y su intensidad (alto / bajo), y apunta al equipo correcto. El u...
4 = Correcta con un matiz menor
    El tipo de hecho y la DIRECCIÓN del impacto son correctos, pero falla la intensidad o el equipo afectado. El usuario decide en el sentido correcto...
3 = No engaña, pero tampoco sirve
    Cae aquí todo lo que deja al usuario como estaba, sin empujarlo a un error. Son dos situaciones. (a) NO EMITE SEÑAL habiendo una: se pierde la not...
2 = Señal engañosa
    Emite una señal donde no la había, o infla el impacto a ALTO cuando no lo era. El usuario actúa sobre información que no sostiene la decisión. Es ...
1 = Señal dañina
    Cualquiera de estas dos. (a) INVIERTE EL SIGNO del impacto: dice negativo donde era positivo o al revés, así que empuja al usuario exactamente en ...


## 3 · ¿El juez sirve? El sanity check

Un juez que le pone el mismo número a todo no es evidencia de nada. Antes de usarlo le pasamos el
**mismo fragmento** con cinco señales de calidad conocida y comprobamos que las ordena.

Esto no es un adorno: la primera versión del prompt **no pasó** —le daba un 5 a una señal con el
signo invertido— y eso guió el diseño. Con las anclas en orden ascendente y un ejemplo resuelto por
nivel pasó de 1/5 a 3/5 aciertos exactos.

In [ ]:
from judge import Judge

juez = Judge()
print(f"Juez: {juez.modelo_id} | rúbrica v{juez.version_rubrica} | device={juez.device}")
res = juez.sanity_check()
for caso, nivel in res["niveles_obtenidos"].items():
    esp = res["niveles_esperados"][caso]
    print(f"  {caso:<20} -> {nivel} (esperado {esp}) conf={res['confianza'][caso]:.2f}")
print(f"\nsepara útil de inútil:   {res['separa_util_de_inutil']}")
print(f"detecta signo invertido: {res['detecta_signo_invertido']}")

Juez: Qwen/Qwen2.5-1.5B-Instruct | rúbrica v1 | device=cpu
  correcta             -> 5 (esperado 5) conf=0.99
  equipo_incorrecto    -> 4 (esperado 4) conf=0.86
  categoria_vecina     -> 4 (esperado 3) conf=0.39
  sin_senal            -> 3 (esperado 3) conf=0.87
  signo_invertido      -> 3 (esperado 1) conf=0.46

separa útil de inútil:   True
detecta signo invertido: False


> **El punto ciego, dicho sin rodeos.** El juez de 1.5B **no** reconoce la inversión de signo: la
> puntúa 3 con una confianza de ~0,46, es decir, dudando. No lo forzamos con más prompt engineering
> ni lo escondemos: ese error concreto —el más caro de nuestra rúbrica— **no se delega en el juez**,
> lo verifica la dimensión 3 de forma determinista. Es la razón arquitectónica de que haya tres
> dimensiones y no dos.

## 4 · El harness: `harness(eval_set, sistema)`

Las tres dimensiones sobre el mismo eval set. El sistema evaluado es cualquier callable que cumpla el
contrato, así que el RAG de M3 entrará sin tocar este archivo:

```python
def sistema(entrada: dict) -> dict:
    #  entrada   {"text", "source", "published_at"}
    #  respuesta {"category", "impact", "team"}
```

In [ ]:
from pathlib import Path

from harness import SistemaLexico, SistemaLoRA, harness, imprimir_scorecard, sistema_mayoritaria

sistemas = [
    ("mayoritaria", sistema_mayoritaria),
    ("lexico", SistemaLexico(Path("../data_collection/lexicon.yaml"))),
    (
        "lora_finetuned",
        SistemaLoRA(
            Path("../m1_lora_adapter_holdout"),
            Path("../data_collection/lexicon.yaml"),
            "dccuchile/bert-base-spanish-wwm-cased",
        ),
    ),
]
tarjetas = [harness(eval_set, s, nombre, juez) for nombre, s in sistemas]
imprimir_scorecard(tarjetas, eval_set, res)

SCORECARD · Offside
Eval set: 21 ejemplos (13 adversariales / borde, 62%)
DIMENSIÓN 1 · MÉTRICA CLÁSICA (automática)
sistema            F1 macro   accuracy
mayoritaria          0.0400     0.1905
lexico               0.4238     0.5238
lora_finetuned       0.3208     0.3810
  accuracy va solo como contraste: sube con la clase mayoritaria.
DIMENSIÓN 2 · LLM-AS-A-JUDGE (rúbrica 1-5 anclada)
sistema            media   distribución 1..5
mayoritaria         2.52   1:3 2:4 3:14 4:0 5:0
lexico              2.57   1:4 2:3 3:13 4:0 5:1
lora_finetuned      2.52   1:3 2:4 3:14 4:0 5:0
  El juez separa útil de inútil: SÍ.
  Detecta el signo invertido:    NO — punto ciego medido.
  Por eso el signo invertido lo verifica la dimensión 3, no el juez.
  Poder discriminante sobre este eval set: nota media 2.83 cuando la categoría era correcta contra 2.38 cuando no lo era (delta +0.45).
  En 9 de 21 ejemplos le da la MISMA nota a una respuesta correcta y a una incorrecta.
DIMENSIÓN 3 · DOMINIO (tasa de señ

## 5 · Sesgo del juez: verbosidad

**El sesgo:** los jueces LLM puntúan más alto las respuestas largas aunque no aporten información.

**Por qué este.** De cara a M3 nos afecta de lleno: el RAG va a redactar señales con más texto que el
clasificador de M1, y si el juez premia la longitud, el RAG «ganaría» sin ser mejor. La comparación
entre módulos —que es todo el punto de tener una vara fija— quedaría rota.

**La detección.** Tomamos las mismas predicciones y las renderizamos con tres longitudes y
exactamente la misma información. Si la nota sube con la longitud, el sesgo está: nada más cambió.

**La mitigación.** `render_senal()` emite todas las señales con la misma plantilla de longitud fija,
sea cual sea el sistema. La longitud deja de ser una variable y el juez no tiene de dónde sacar la
preferencia. Es estructural, no un ajuste sobre el puntaje.

In [ ]:
!python bias_check.py

SESGO DE VERBOSIDAD DEL JUEZ
condición     long. media   nota media
escueta                24         2.62
fija                   55         2.57
verbosa               352         3.05
  delta verbosa - escueta: +0.43 puntos de rúbrica
  ejemplos que cambian de nivel solo por la longitud: 7/21
  ¿Sesgo detectado? SÍ — el juez premia la longitud sin más información
  MITIGACIÓN: el harness emite todas las señales con la plantilla FIJA
  (`render_senal`), así que ningún sistema puede ganar puntos escribiendo
  más largo. Es la condición del medio de esta tabla.
Informe -> C:\Users\USUARIO\Desktop\Transformers\proyecto1\eval\bias_report.json

## 6 · Lectura honesta

Ver [`../docs/harness-m2.md`](../docs/harness-m2.md) para el análisis completo y
[`../README.md`](../../README.md) para el resumen.

*SI4006 · Universidad EAFIT · Módulo 2 · Equipo Offside*